In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

In [ ]:
def scrape_bankfive_reviews(total_pages=6):
    all_reviews = []
    base_url = "https://www.trustpilot.com/review/www.bankfive.com"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    }

    print(f"Starting data collection for BankFive...")

    for page in range(1, total_pages + 1):
        url = f"{base_url}?page={page}"
        print(f"Scraping Page {page}...")
        
        try:
            response = requests.get(url, headers=headers)
            if response.status_code != 200:
                print(f"Failed to fetch page {page}")
                continue
                
            soup = BeautifulSoup(response.text, 'html.parser')
            
            data_script = soup.find("script", id="__NEXT_DATA__")
            
            if data_script:
                json_data = json.loads(data_script.string)
                reviews_list = json_data['props']['pageProps']['reviews']
                
                for r in reviews_list:
                    review_text = r.get('text', '')
                    review_title = r.get('title', '')
                    rating = r.get('rating', '')
                    
                    # Combine title and text for a richer sentiment record
                    full_content = f"{review_title}. {review_text}".strip()
                    
                    if len(full_content) > 10:  
                        all_reviews.append({
                            "source": "Trustpilot",
                            "rating": rating,
                            "text": full_content
                        })
            
            time.sleep(1)
            
        except Exception as e:
            print(f"An error occurred on page {page}: {e}")

    return all_reviews

In [ ]:
data = scrape_bankfive_reviews(total_pages=7) 

Starting data collection for BankFive...
Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Scraping Page 6...
Scraping Page 7...


In [14]:
df = pd.DataFrame(data)
df

,source,rating,text
0,Trustpilot,5,The online rewards and reviews for bank 5\n. M...
1,Trustpilot,3,Wait time to meet with someone was a…. Wait ti...
2,Trustpilot,5,i had very good and smooth transection…. i had...
3,Trustpilot,5,Every time that i need help i called…. Every t...
4,Trustpilot,5,Teller was very efficient and…. Teller was ver...
...,...,...,...
135,Trustpilot,5,Friendly & quick service. Friendly & quick ser...
136,Trustpilot,4,Great experience thanks. Great experience thanks
137,Trustpilot,3,I still haven't received my new debit…. I stil...
138,Trustpilot,1,When I sign on on my bank account with…. When ...


In [16]:
df = df.drop_duplicates(subset=['text'])
df

,source,rating,text
0,Trustpilot,5,The online rewards and reviews for bank 5\n. M...
1,Trustpilot,3,Wait time to meet with someone was a…. Wait ti...
2,Trustpilot,5,i had very good and smooth transection…. i had...
3,Trustpilot,5,Every time that i need help i called…. Every t...
4,Trustpilot,5,Teller was very efficient and…. Teller was ver...
...,...,...,...
135,Trustpilot,5,Friendly & quick service. Friendly & quick ser...
136,Trustpilot,4,Great experience thanks. Great experience thanks
137,Trustpilot,3,I still haven't received my new debit…. I stil...
138,Trustpilot,1,When I sign on on my bank account with…. When ...


In [17]:
csv_filename = "bankfive_sentiment_data.csv"
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"Data saved to {csv_filename}")

Data saved to bankfive_sentiment_data.csv


In [ ]:
df_verified = pd.read_csv("bankfive_sentiment_data.csv")

print(f"Total records collected: {len(df_verified)}")

print("\nDistribution of Ratings:")
print(df_verified['rating'].value_counts().sort_index())

print("\nMissing values per column:")
print(df_verified.isnull().sum())

Total records collected: 140

Distribution of Ratings:
rating
1      4
3      5
4     11
5    120
Name: count, dtype: int64

Missing values per column:
source    0
rating    0
text      0
dtype: int64


In [ ]:
print("Top 10 Records:")
display(df_verified.head(10))

print("\nBottom 10 Records:")
display(df_verified.tail(10))

Top 10 Records:


,source,rating,text
0,Trustpilot,5,The online rewards and reviews for bank 5\n. M...
1,Trustpilot,3,Wait time to meet with someone was a…. Wait ti...
2,Trustpilot,5,i had very good and smooth transection…. i had...
3,Trustpilot,5,Every time that i need help i called…. Every t...
4,Trustpilot,5,Teller was very efficient and…. Teller was ver...
5,Trustpilot,5,Very friendly staff and great knowledge . The ...
6,Trustpilot,5,Friendly people that know your name. Christina...
7,Trustpilot,5,Great experience!. Everyone was very friendly ...
8,Trustpilot,5,Great experience staff cooperative and…. Great...
9,Trustpilot,5,Great employees. The employees are courteous a...



Bottom 10 Records:


,source,rating,text
130,Trustpilot,5,Very nice and effiient. Very nice and effiient
131,Trustpilot,5,Great Bank. I have used this bank for over 20 ...
132,Trustpilot,5,Love BankFive. Love BankFive! Everyone is so f...
133,Trustpilot,5,Using in Portugal. I was in Europe and had no ...
134,Trustpilot,5,It is a great bank. The people who work there
135,Trustpilot,5,Friendly & quick service. Friendly & quick ser...
136,Trustpilot,4,Great experience thanks. Great experience thanks
137,Trustpilot,3,I still haven't received my new debit…. I stil...
138,Trustpilot,1,When I sign on on my bank account with…. When ...
139,Trustpilot,5,A senior employee worked me through the…. A se...


In [ ]:
df_verified['word_count'] = df_verified['text'].apply(lambda x: len(str(x).split()))
print(f"Average words per review: {df_verified['word_count'].mean():.2f}")
print(f"Longest review: {df_verified['word_count'].max()} words")

Average words per review: 21.41
Longest review: 86 words


# Week 2

In [ ]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
import emoji

# Download necessary NLTK data
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

In [ ]:
df = pd.read_csv("bankfive_sentiment_data.csv")
df